In [1]:
%pip install pandas scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [8]:

import pandas as pd
import os
import glob

# ==========================================
# 1. Dataset folder and file path
# ==========================================

folder_path = r"D:\real think\AI&ML INTERNSHIP"

file_path = os.path.join(
    folder_path,
    "Bitext_Sample_Customer_Service_Testing_Dataset.csv"
)

# ==========================================
# 2. Check whether the folder exists
# ==========================================

if not os.path.isdir(folder_path):
    raise FileNotFoundError(
        f"Dataset folder not found: {folder_path}"
    )

# ==========================================
# 3. Find customer service CSV files
# ==========================================

files = glob.glob(
    os.path.join(folder_path, "*Customer*Service*.csv")
)

if not files:
    raise FileNotFoundError(
        "No customer service CSV files found in the folder."
    )

print("Files found:")
for file in files:
    print(os.path.basename(file))

# ==========================================
# 4. Load the required dataset
# ==========================================

if not os.path.isfile(file_path):
    raise FileNotFoundError(
        f"Required dataset not found: {file_path}"
    )

df = pd.read_csv(file_path)

print("\nDataset loaded successfully!")
print("Shape before removing duplicates:", df.shape)

# ==========================================
# 5. Check required columns
# ==========================================

required_columns = {"utterance", "intent", "category", "tags"}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Required columns missing: {missing_columns}"
    )

print("\nColumn names:", df.columns.tolist())

# ==========================================
# 6. Check duplicate rows
# ==========================================

duplicate_count = df.duplicated().sum()

print("\nDuplicate rows found:", duplicate_count)

df = df.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", df.shape)

# ==========================================
# 7. Check missing values
# ==========================================

print("\nMissing values in each column:")
print(df.isnull().sum())

# ==========================================
# 8. Check empty text values
# ==========================================

for column in ["utterance", "intent", "category", "tags"]:
    if df[column].dtype == "object":
        df[column] = df[column].replace(
            r"^\s*$", pd.NA, regex=True
        )

print("\nMissing values after checking empty strings:")
print(df.isnull().sum())

# ==========================================
# 9. Dataset overview
# ==========================================

print("\nDataset information:")
df.info()

print("\nNumber of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nUnique intents:", df["intent"].nunique())
print("Unique categories:", df["category"].nunique())
print("Unique tags:", df["tags"].nunique())

# ==========================================
# 10. Intent distribution
# ==========================================

print("\nIntent distribution:")
print(df["intent"].value_counts())

# ==========================================
# 11. Category distribution
# ==========================================

print("\nCategory distribution:")
print(df["category"].value_counts())

# ==========================================
# 12. Display first five rows
# ==========================================

print("\nFirst five rows:")
display(df.head())

# ==========================================
# 13. Display statistical overview
# ==========================================

print("\nDataset summary:")
display(df.describe(include="all").T)

# ==========================================
# 14. Save cleaned dataset
# ==========================================

output_path = os.path.join(
    folder_path,
    "Bitext_Customer_Service_Cleaned.csv"
)

df.to_csv(output_path, index=False)

print("\nCleaned dataset saved successfully!")
print("Output file:", output_path)


Files found:
Bitext_Sample_Customer_Service_Testing_Dataset.csv

Dataset loaded successfully!
Shape before removing duplicates: (818, 4)

Column names: ['utterance', 'intent', 'category', 'tags']

Duplicate rows found: 0
Shape after removing duplicates: (818, 4)

Missing values in each column:
utterance    0
intent       0
category     0
tags         0
dtype: int64

Missing values after checking empty strings:
utterance    0
intent       0
category     0
tags         0
dtype: int64

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 818 entries, 0 to 817
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   utterance  818 non-null    str  
 1   intent     818 non-null    str  
 2   category   818 non-null    str  
 3   tags       818 non-null    str  
dtypes: str(4)
memory usage: 80.1 KB

Number of rows: 818
Number of columns: 4

Unique intents: 27
Unique categories: 11
Unique tags: 61

Intent distribution:
intent
cont

,utterance,intent,category,tags
0,I have a question about cancelling an order,cancel_order,ORDER,B
1,help canceling the order I have made,cancel_order,ORDER,B
2,I do not know how to cancel an order I have made,cancel_order,ORDER,BE
3,where can I cancel the last order I made?,cancel_order,ORDER,BI
4,I don't want the last order I made,cancel_order,ORDER,B



Dataset summary:


,count,unique,top,freq
utterance,818,818,I have a question about cancelling an order,1
intent,818,27,contact_human_agent,42
category,818,11,ACCOUNT,172
tags,818,61,B,261



Cleaned dataset saved successfully!
Output file: D:\real think\AI&ML INTERNSHIP\Bitext_Customer_Service_Cleaned.csv


In [10]:
# misvalue 
df = df[["utterance", "intent"]].dropna().copy()
df = df.drop_duplicates()

X = df["utterance"].astype(str)
y = df["intent"].astype(str)


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)
print("Training complete")




Training complete


In [12]:
from sklearn.metrics import classification_report, confusion_matrix

pred = model.predict(X_test)

print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

                          precision    recall  f1-score   support

            cancel_order       1.00      1.00      1.00         5
            change_order       0.88      0.88      0.88         8
 change_shipping_address       1.00      1.00      1.00         8
  check_cancellation_fee       0.80      0.80      0.80         5
           check_invoice       0.71      0.83      0.77         6
   check_payment_methods       1.00      1.00      1.00         5
     check_refund_policy       1.00      1.00      1.00         8
               complaint       1.00      1.00      1.00         5
contact_customer_service       1.00      1.00      1.00         5
     contact_human_agent       1.00      1.00      1.00         9
          create_account       0.75      0.60      0.67         5
          delete_account       1.00      0.83      0.91         6
        delivery_options       1.00      1.00      1.00         6
         delivery_period       0.80      1.00      0.89         8
         

In [13]:
message = input("Enter customer message: ")
print("Predicted intent:", model.predict([message])[0])

Enter customer message:  exit


Predicted intent: change_order
